# Reproducing SimPO (Llama-3-Instruct-8B-SimPO inference)

**Paper:** [arXiv:2405.14734](https://arxiv.org/abs/2405.14734) — Meng, Xia, Chen, NeurIPS 2024
**Repository:** [princeton-nlp/SimPO](https://github.com/princeton-nlp/SimPO)
**Pretrained:** princeton-nlp/Llama-3-Instruct-8B-SimPO (paper Table 4 best config)

## Honest scope of this run

- Hardware: Kaggle free-tier T4/P100 16GB, 9h session
- Model: **Llama-3-Instruct-8B-SimPO** loaded in **int4** (16GB tight; bf16 only on Ampere+)
- Protocol: **inference-only with pretrained checkpoint**, greedy decoding
- Prompts: **20 AlpacaEval-style prompts**, max_new_tokens=128
- We do NOT claim paper's AlpacaEval-2 LC win-rate (44.7%) or Arena-Hard (33.8%) — those need GPT-4 judge.
- We measure: model params, response length distribution, perplexity on 10 self-generated continuations.


## 1. Setup

In [ ]:
import os, time, json, sys, subprocess

t0 = time.time()

subprocess.run(["pip", "install", "-q", "--upgrade",
                "torch==2.4.1",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

subprocess.run(["pip", "install", "-q",
                "transformers>=4.43", "accelerate>=0.27", "bitsandbytes>=0.43", "sentencepiece", "protobuf<4"], check=True)

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} device={gpu_name} sm={gpu_cap}")
print(f"setup elapsed: {time.time()-t0:.1f}s")

USE_BF16 = gpu_cap >= (8, 0)
USE_INT4 = True  # default safer for 16GB
print(f"USE_BF16={USE_BF16} USE_INT4={USE_INT4}")


## 2. Load Llama-3-Instruct-8B-SimPO

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "princeton-nlp/Llama-3-Instruct-8B-SimPO"

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

kwargs = dict(low_cpu_mem_usage=True, device_map="auto")
if USE_INT4:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    kwargs["quantization_config"] = bnb_config
else:
    kwargs["torch_dtype"] = torch.bfloat16 if USE_BF16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
model.eval()

# Architectural param count from config (4bit Params4bit obscures p.numel()).
cfg = model.config
h = cfg.hidden_size; v = cfg.vocab_size; n_layers = cfg.num_hidden_layers
intermed = cfg.intermediate_size
embed = v * h
per_layer = 4 * h * h + 3 * h * intermed + 2 * h
n_params = embed + n_layers * per_layer + h
print(f"Llama-3-Instruct-8B-SimPO loaded: ~{n_params/1e9:.2f}B params (paper architectural ~8.0B)")
print(f"load elapsed: {time.time()-t0:.1f}s")
print(f"GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 3. Prepare 20 AlpacaEval-style prompts

Llama-3-Instruct chat template applied via tokenizer.


In [ ]:
PROMPTS = [
    "What are the names of some famous actors that started their careers on Broadway?",
    "How did US states get their names?",
    "Hi, my sister and her girlfriends want me to play kickball with them. Can you explain how the game is played, so they don't take advantage of me?",
    "What is some cool music from the 1920s?",
    "How do I wrap a present neatly?",
    "How do I dice without slicing my finger?",
    "Can you explain what the Big Bang theory is?",
    "What is a polygon?",
    "How do I take care of a wooden table?",
    "What were the major contributions of the ancient Egyptians to mathematics?",
    "What are some good books to read for someone learning about machine learning?",
    "Can you give me a simple recipe for chicken curry?",
    "Who painted the Mona Lisa?",
    "What is the capital of Australia?",
    "Explain quantum entanglement in simple terms.",
    "What are the benefits of meditation?",
    "How does photosynthesis work?",
    "What is the difference between weather and climate?",
    "List 3 tips for learning a new language.",
    "What is the speed of light in a vacuum?",
]
MAX_NEW_TOKENS = 128


## 4. Generate responses (greedy)

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
responses = []
gen_t0 = time.time()

for i, prompt in enumerate(PROMPTS):
    messages = [{"role": "user", "content": prompt}]
    chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(chat, return_tensors="pt").input_ids.to(model.device)

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            input_ids, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=1.0, top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - t0

    new_tokens = out.shape[1] - input_ids.shape[1]
    text = tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    responses.append({"prompt": prompt, "response": text, "new_tokens": int(new_tokens), "elapsed_s": elapsed})
    if i < 3:
        print(f"  [{i}] {new_tokens} tok / {elapsed:.2f}s — {text[:120]!r}")

gen_elapsed = time.time() - gen_t0
all_lens = [r["new_tokens"] for r in responses]
import numpy as np
resp_len_mean = float(np.mean(all_lens))
resp_len_std = float(np.std(all_lens))
resp_len_median = float(np.median(all_lens))
print(f"\ngen aggregate: {sum(all_lens)} tok / {gen_elapsed:.1f}s")
print(f"response length: mean={resp_len_mean:.1f} std={resp_len_std:.1f} median={resp_len_median:.1f}")
gen_peak_gb = float(torch.cuda.max_memory_allocated() / 1e9)


## 5. Perplexity on 10 self-generated continuations

In [ ]:
import math
ppls = []
for i in range(min(10, len(responses))):
    full_text = responses[i]["prompt"] + " " + responses[i]["response"]
    enc = tokenizer(full_text, return_tensors="pt").input_ids.to(model.device)
    if enc.shape[1] < 5:
        continue
    with torch.inference_mode():
        out = model(enc, labels=enc)
        loss = float(out.loss.item())
    ppl = math.exp(loss) if loss < 50 else float("inf")
    ppls.append(ppl)
    if i < 3:
        print(f"  [{i}] loss={loss:.3f} ppl={ppl:.2f}")

valid_ppls = [p for p in ppls if math.isfinite(p)]
ppl_mean = float(np.mean(valid_ppls)) if valid_ppls else float("inf")
ppl_median = float(np.median(valid_ppls)) if valid_ppls else float("inf")
print(f"\nperplexity over {len(valid_ppls)} prompts: mean={ppl_mean:.2f} median={ppl_median:.2f}")


## 6. Write metrics.json

In [ ]:
measured = {}
measured["pipeline_verified"] = 1.0 if (resp_len_mean > 5 and len(valid_ppls) >= 5) else 0.0
measured["model_params_b"] = float(n_params / 1e9)
measured["prompts_generated"] = float(len(responses))
measured["max_new_tokens"] = float(MAX_NEW_TOKENS)
measured["response_len_mean"] = resp_len_mean
measured["response_len_std"] = resp_len_std
measured["response_len_median"] = resp_len_median
measured["gen_time_s"] = float(gen_elapsed)
measured["gen_tokens_per_sec"] = float(sum(all_lens) / gen_elapsed) if gen_elapsed > 0 else 0.0
measured["gen_gpu_peak_gb"] = gen_peak_gb
measured["perplexity_mean"] = ppl_mean
measured["perplexity_median"] = ppl_median
measured["perplexity_n"] = float(len(valid_ppls))
measured["used_int4"] = 1.0 if USE_INT4 else 0.0
measured["used_bf16"] = 1.0 if USE_BF16 else 0.0

paper_reference = {
    "model": "Llama-3-Instruct-8B-SimPO",
    "alpacaeval2_lc_winrate_paper": 44.7,
    "arena_hard_winrate_paper": 33.8,
    "params_b_paper": 8.0,
    "note": "AlpacaEval-2 LC + Arena-Hard win rates require GPT-4 judge. Out of scope for free-tier inference run.",
}

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
with open(os.path.join(out_dir, "metrics.json"), "w") as f:
    json.dump(measured, f, indent=2)
with open(os.path.join(out_dir, "paper_reference.json"), "w") as f:
    json.dump(paper_reference, f, indent=2)
with open(os.path.join(out_dir, "responses.json"), "w") as f:
    json.dump(responses, f, indent=2)

print("=== measured ===")
print(json.dumps(measured, indent=2))
print("=== paper_reference ===")
print(json.dumps(paper_reference, indent=2))


## Appendix — what this run does and does not show

**Shows:**
- princeton-nlp/Llama-3-Instruct-8B-SimPO loads and runs inference on free-tier 16GB GPU (int4 by default).
- Architectural param count (~8B) consistent with paper.
- 20-prompt response generation completes; response length distribution is non-degenerate.
- 10-prompt perplexity finite and within sane LLM range.

**Does NOT show:**
- Paper's AlpacaEval-2 LC win-rate (44.7%) — requires GPT-4 judge, not run.
- Arena-Hard win-rate (33.8%) — requires GPT-4 judge.
- DPO / KTO / IPO baseline comparisons — single-model run.
- bf16/fp16 vs int4 numerical equivalence — int4 is a quant approximation by default.

**Expected verdict:** `partial` — pipeline + architectural param verified; paper win-rate metrics NOT independently measured.
